# Tesseract Baseline

Use this notebook only to orchestrate Tesseract baseline runs.
Import reusable logic from `src/`; do not implement business logic here.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
DEFAULT_COLAB_ROOT = Path(os.environ.get('NFSE_PROJECT_ROOT', '/content/nfse-extractor'))
ROOT = DEFAULT_COLAB_ROOT if IS_COLAB else (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG = {
    'bootstrap_runtime': True,
    'mount_drive': False,
    'sample_path': '',
    'output_path': str(ROOT / 'artifacts' / 'tesseract_raw_artifacts.json'),
}


In [ ]:
if IS_COLAB and CONFIG['bootstrap_runtime']:
    subprocess.run(['bash', str(ROOT / 'scripts' / 'colab_bootstrap.sh')], check=True)

if IS_COLAB and CONFIG['mount_drive']:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
from src.engines import TesseractExtractionAdapter
from src.export import write_extracted_elements_json
from src.ingestion import load_document
from src.preprocessing import preprocess_document

if not CONFIG['sample_path']:
    raise ValueError('Set CONFIG[\'sample_path\'] to a sample image path before running the pipeline.')

document = load_document(CONFIG['sample_path'])
preprocessed = preprocess_document(document)
adapter = TesseractExtractionAdapter()
artifacts = adapter.extract_preprocessed(preprocessed)
output_path = write_extracted_elements_json(artifacts, CONFIG['output_path'])

print(f'Document: {document.document_id}')
print(f'Pages: {preprocessed.metadata["page_count"]}')
print(f'Artifacts: {len(artifacts)}')
print(f'Output: {output_path}')
